# Grammar Scoring Engine — Speech Preprocessing Pipeline

**Goal:** prepare 45–60 s spoken audio clips so a model can predict a **grammar MOS score (0–5)**.

**Why preprocessing matters here:** grammar is judged from *what* the speaker says, so the most important output of this stage is a **faithful transcript**. Every audio step below is chosen to help speech recognition (Whisper) hear the speaker clearly **without "correcting" their mistakes**. We also extract **fluency features** (pauses, speech rate) because human raters are influenced by them.

| Step | Method | Why |
|---|---|---|
| 1 | Data discovery & integrity check | Catch missing/corrupt files before training |
| 2 | Exploratory analysis | Know label distribution and clip durations |
| 3 | Resample to 16 kHz mono | Required input format for Whisper / wav2vec2 |
| 4 | DC-offset removal | Removes microphone bias that distorts energy features |
| 5 | Leading/trailing silence trimming | Removes dead air without touching in-speech pauses |
| 6 | Optional noise reduction | Cleans noisy recordings (applied only when SNR is low) |
| 7 | Loudness normalisation | Makes quiet and loud speakers comparable |
| 8 | Quality metrics (SNR, clipping, speech ratio) | Flags bad recordings; useful as features |
| 9 | Voice-activity / pause analysis | Fluency features: pauses, speech ratio |
| 10 | Verbatim transcription (faster-whisper) | Turns speech into text **keeping errors and fillers** |
| 11 | Transcript cleaning & text features | Word count, speech rate, fillers, repetitions |
| 12 | Stratified CV folds | Fair evaluation on only 769 samples |



In [3]:
# !pip install -q faster-whisper noisereduce pyloudnorm

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
apache-airflow-core 3.0.3 requires termcolor>=3.0.0, but you have termcolor 2.4.0 which is incompatible.
dagshub 0.6.2 requires dacite~=1.6.0, but you have dacite 1.9.2 which is incompatible.
streamlit 1.37.1 requires protobuf<6,>=3.20, but you have protobuf 6.31.1 which is incompatible.
tensorflow 2.19.0 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<6.0.0dev,>=3.20.3, but you have protobuf 6.31.1 which is incompatible.


In [7]:
%pip install librosa soundfile tqdm matplotlib

  Using cached numpy-2.5.3-cp312-cp312-win_amd64.whl.metadata (6.6 kB)
  Using cached scipy-1.18.1-cp312-cp312-win_amd64.whl.metadata (61 kB)
  Using cached scikit_learn-1.9.1-cp312-cp312-win_amd64.whl.metadata (9.3 kB)
INFO: pip is looking at multiple versions of contourpy to determine which version is compatible with other requirements. This could take a while.
  Using cached contourpy-1.4.0-cp312-cp312-win_amd64.whl.metadata (3.8 kB)
  Using cached narwhals-2.26.0-py3-none-any.whl.metadata (15 kB)
   ---------------------------------------- 0.0/1.0 MB ? eta -:--:--
   ------------------------------ --------- 0.8/1.0 MB 4.2 MB/s eta 0:00:01
   ---------------------------------------- 1.0/1.0 MB 3.7 MB/s eta 0:00:00
Using cached contourpy-1.4.0-cp312-cp312-win_amd64.whl (234 kB)
   ---------------------------------------- 0.0/2.8 MB ? eta -:--:--
   -------------- ------------------------- 1.0/2.8 MB 5.6 MB/s eta 0:00:01
   ----------------------------- ---------- 2.1/2.8 MB 5.1 MB/s 

  You can safely remove it manually.
  You can safely remove it manually.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gensim 4.3.3 requires numpy<2.0,>=1.18.5, but you have numpy 2.5.3 which is incompatible.
gensim 4.3.3 requires scipy<1.14.0,>=1.7.0, but you have scipy 1.18.1 which is incompatible.
streamlit 1.37.1 requires protobuf<6,>=3.20, but you have protobuf 6.31.1 which is incompatible.
tensorflow 2.19.0 requires numpy<2.2.0,>=1.26.0, but you have numpy 2.5.3 which is incompatible.
tensorflow 2.19.0 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<6.0.0dev,>=3.20.3, but you have protobuf 6.31.1 which is incompatible.
ydata-profiling 4.16.1 requires numba<=0.61,>=0.56.0, but you have numba 0.68.0 which is incompatible.
ydata-profiling 4.16.1 requires numpy<2.2,>=1.16.0, but you have numpy 2.5.3 which is incompatible.
ydata-prof

In [9]:
import os, glob, re, json, warnings
import numpy as np
import pandas as pd
import librosa
import soundfile as sf
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")
SEED = 42
np.random.seed(SEED)

TARGET_SR = 16000                      # Whisper / wav2vec2 expect 16 kHz

# ---- Local paths (Windows) ----
PROJECT_DIR = r"C:\Users\yashw\SSHL\shl-hiring-assessment-2026"
DATA_DIR    = os.path.join(PROJECT_DIR, "Dataset_Final")   # your dataset
WORK_DIR    = os.path.join(PROJECT_DIR, "working")         # replaces /kaggle/working
CLEAN_DIR   = os.path.join(WORK_DIR, "clean_audio")
os.makedirs(CLEAN_DIR, exist_ok=True)

# Quick check that the dataset is found
assert os.path.isdir(DATA_DIR), f"Dataset folder not found: {DATA_DIR}"
print("Dataset folder:", DATA_DIR)
print("Contents:", os.listdir(DATA_DIR)[:20])

# Find all audio files (any subfolder)
audio_files = [f for ext in ("wav", "mp3", "flac", "m4a", "ogg")
               for f in glob.glob(os.path.join(DATA_DIR, "**", f"*.{ext}"), recursive=True)]
print(f"Found {len(audio_files)} audio files")

Dataset folder: C:\Users\yashw\SSHL\shl-hiring-assessment-2026\Dataset_Final
Contents: ['sample_submission.csv', 'test', 'test.csv', 'train', 'train.csv']
Found 985 audio files


In [11]:
train_df = pd.read_csv(os.path.join(DATA_DIR, "train.csv"))
test_df  = pd.read_csv(os.path.join(DATA_DIR, "test.csv"))

print("Train:", train_df.shape)
print("Test: ", test_df.shape)
display(train_df.head())
display(test_df.head())

Train: (769, 2)
Test:  (216, 2)


,filename,label
0,audio_192.wav,3.0
1,audio_436.wav,3.0
2,audio_447.wav,3.5
3,audio_126.wav,2.0
4,audio_61.wav,2.0


,filename,label
0,audio_128.wav,-1
1,audio_156.wav,-1
2,audio_19.wav,-1
3,audio_108.wav,-1
4,audio_206.wav,-1


In [13]:
for root, dirs, files in os.walk(DATA_DIR):
    level = root.replace(DATA_DIR, "").count(os.sep)
    print("  " * level + os.path.basename(root) + "/")
    for f in files[:5]:                      # first 5 files per folder
        print("  " * (level + 1) + f)
    if len(files) > 5:
        print("  " * (level + 1) + f"... ({len(files)} files)")

Dataset_Final/
  sample_submission.csv
  test.csv
  train.csv
  test/
    audio_0.wav
    audio_1.wav
    audio_10.wav
    audio_100.wav
    audio_101.wav
    ... (216 files)
  train/
    audio_0.wav
    audio_1.wav
    audio_10.wav
    audio_100.wav
    audio_101.wav
    ... (769 files)


In [15]:
# ---- Locate train / test audio folders ----
AUDIO_EXTS = (".wav", ".mp3", ".flac", ".m4a", ".ogg")

def find_audio_dir(split):
    """Return the folder under DATA_DIR whose name contains `split` and holds audio files."""
    candidates = []
    for root, dirs, files in os.walk(DATA_DIR):
        n_audio = sum(f.lower().endswith(AUDIO_EXTS) for f in files)
        if n_audio and split in os.path.basename(root).lower():
            candidates.append((n_audio, root))
    if not candidates:
        raise FileNotFoundError(f"No '{split}' audio folder found under {DATA_DIR}")
    return max(candidates)[1]          # pick the folder with the most audio files

TRAIN_AUDIO_DIR = find_audio_dir("train")
TEST_AUDIO_DIR  = find_audio_dir("test")
print("Train audio:", TRAIN_AUDIO_DIR, "->", len(os.listdir(TRAIN_AUDIO_DIR)), "files")
print("Test audio: ", TEST_AUDIO_DIR,  "->", len(os.listdir(TEST_AUDIO_DIR)),  "files")

# ---- Load CSVs ----
train_df = pd.read_csv(os.path.join(DATA_DIR, "train.csv"))
test_df  = pd.read_csv(os.path.join(DATA_DIR, "test.csv"))
print("Train CSV columns:", list(train_df.columns))
print("Test CSV columns: ", list(test_df.columns))

# ---- Attach full audio paths to each row ----
FILE_COL = next(c for c in train_df.columns if "file" in c.lower() or "name" in c.lower())

def to_path(fname, folder):
    fname = str(fname)
    if not fname.lower().endswith(AUDIO_EXTS):
        fname += ".wav"                # CSV may store names without extension
    return os.path.join(folder, fname)

train_df["path"] = train_df[FILE_COL].apply(lambda f: to_path(f, TRAIN_AUDIO_DIR))
test_df["path"]  = test_df[FILE_COL].apply(lambda f: to_path(f, TEST_AUDIO_DIR))

# ---- Sanity check: every row has a real file ----
train_missing = (~train_df["path"].apply(os.path.exists)).sum()
test_missing  = (~test_df["path"].apply(os.path.exists)).sum()
print(f"Missing train files: {train_missing} / {len(train_df)}")
print(f"Missing test files:  {test_missing} / {len(test_df)}")

display(train_df.head())

Train audio: C:\Users\yashw\SSHL\shl-hiring-assessment-2026\Dataset_Final\train -> 769 files
Test audio:  C:\Users\yashw\SSHL\shl-hiring-assessment-2026\Dataset_Final\test -> 216 files
Train CSV columns: ['filename', 'label']
Test CSV columns:  ['filename', 'label']
Missing train files: 0 / 769
Missing test files:  0 / 216


,filename,label,path
0,audio_192.wav,3.0,C:\Users\yashw\SSHL\shl-hiring-assessment-2026...
1,audio_436.wav,3.0,C:\Users\yashw\SSHL\shl-hiring-assessment-2026...
2,audio_447.wav,3.5,C:\Users\yashw\SSHL\shl-hiring-assessment-2026...
3,audio_126.wav,2.0,C:\Users\yashw\SSHL\shl-hiring-assessment-2026...
4,audio_61.wav,2.0,C:\Users\yashw\SSHL\shl-hiring-assessment-2026...


In [18]:
%pip install imageio-ffmpeg

   ---------------------------------------- 0.0/31.2 MB ? eta -:--:--
   ---------------------------------------- 0.3/31.2 MB ? eta -:--:--
   - -------------------------------------- 1.3/31.2 MB 4.8 MB/s eta 0:00:07
   -- ------------------------------------- 2.1/31.2 MB 4.7 MB/s eta 0:00:07
   ---- ----------------------------------- 3.4/31.2 MB 4.8 MB/s eta 0:00:06
   ----- ---------------------------------- 4.2/31.2 MB 4.6 MB/s eta 0:00:06
   ------ --------------------------------- 5.0/31.2 MB 4.4 MB/s eta 0:00:06
   ------- -------------------------------- 5.8/31.2 MB 4.3 MB/s eta 0:00:06
   -------- ------------------------------- 6.6/31.2 MB 4.3 MB/s eta 0:00:06
   --------- ------------------------------ 7.6/31.2 MB 4.3 MB/s eta 0:00:06
   ---------- ----------------------------- 8.4/31.2 MB 4.2 MB/s eta 0:00:06
   ----------- ---------------------------- 9.2/31.2 MB 4.2 MB/s eta 0:00:06
   ------------- -------------------------- 10.2/31.2 MB 4.2 MB/s eta 0:00:06
   ---------

In [ ]:
import imageio_ffmpeg
os.environ["PATH"] += os.pathsep + os.path.dirname(imageio_ffmpeg.get_ffmpeg_exe())